ST 554 HW#10\
Hui Fang\
4/19/2026

# Part 1 - Creating Streaming Data Using Rate

Setup a data stream using the `"rate"` format.
Prior to starting the stream, set up a sequence of actions using appropriate functions from `pyspark.sql.functions`
that uses the `rate` data to 
- find the square root of the rate ‘value’ 
- find mod 4 of the rate ‘value’

To output this, create a `writeStream` that writes to ‘memory’ (`format("memory")`). Give the query a name
(`queryName("...")`) and start it!\
Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the queryname (`spark.sql("select * from you_table_name").show()`).

## 1. Create the streaming DataFrame

I first create a Spark Session and create a streaming DataFrame using the built-in 'rate' source. This generates rows with (timestemp, value) at 1 row per second.

In [13]:
# import module
from pyspark.sql import SparkSession
# create a spark session
spark = SparkSession.builder.getOrCreate()
# create a streaming DataFrame using the built-in 'rate' source
rate_df = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 1)
        .load()
)

## 2. Apply transformations

Before starting the stream, I applied transformation from `pyspark.sql.functions` to compute the square root of the 'value' column and the value modulo 4, which is a math operation that gives us the remainder when a number is divided by 4.

In [14]:
# import module
from pyspark.sql.functions import sqrt, col
# apply the required transformations
transformed = (
    rate_df
        .withColumn("sqrt_value", sqrt(col("value")))
        .withColumn("mod4_value", col("value") % 4)
)

## 3. Write the stream to memory

To store the output, I created a `writeStream` that writes to memory useing `format("memory")`. I named the the query "rate_table" and started the stream.

In [15]:
query = (
    transformed.writeStream
               .format("memory")
               .queryName("rate_table")
               .outputMode("append")
               .start()
)

26/04/19 16:28:30 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d4378bab-7426-4d5c-8569-3b787844c358. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 16:28:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 4. Stop the stream and query the in-memory table

After letting the stream run for about 30 seconds, I stopped the query and then used `spark.sql("select * from rate_table").show()` to display all rows stroed in the in-memory table. 

In [16]:
# stop the steam
query.stop()

# query the in-memory table
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-19 16:28:...|    0|               0.0|         0|
|2026-04-19 16:28:...|    1|               1.0|         1|
|2026-04-19 16:28:...|    2|1.4142135623730951|         2|
|2026-04-19 16:28:...|    3|1.7320508075688772|         3|
|2026-04-19 16:28:...|    4|               2.0|         0|
|2026-04-19 16:28:...|    5|  2.23606797749979|         1|
|2026-04-19 16:28:...|    6| 2.449489742783178|         2|
|2026-04-19 16:28:...|    7|2.6457513110645907|         3|
|2026-04-19 16:28:...|    8|2.8284271247461903|         0|
|2026-04-19 16:28:...|    9|               3.0|         1|
|2026-04-19 16:28:...|   10|3.1622776601683795|         2|
|2026-04-19 16:28:...|   11|   3.3166247903554|         3|
|2026-04-19 16:28:...|   12|3.4641016151377544|         0|
|2026-04-19 16:28:...|   13| 3.605551275463989|         

26/04/19 16:29:01 WARN DAGScheduler: Failed to cancel job group f1713df1-d307-4442-8fbc-2ad10fb9fd04. Cannot find active jobs for it.
26/04/19 16:29:01 WARN DAGScheduler: Failed to cancel job group f1713df1-d307-4442-8fbc-2ad10fb9fd04. Cannot find active jobs for it.


# Part 2 - Using data from a CSV with a Pipeline

There are six `bikeDetails` sub datasets available on the assignment link. The one named `bikeDetails_for_fit.csv`
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following
- use an SQLTransformer with the following statement (this does some log transforms, renames a variable, and creates a dummy variable from categorical variable):

`SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__`

- use a `VectorAssembler` to create a `features` column. The `features` column should include the `year`,
`log_km_driven`, and `one_owner` variables.
- create a `Pipeline` with the two steps above (`SQLTransformer` then `VectorAssembler`)
- fit this pipeline to the SQL data frame and save this as an object.

Now we want to set up a read stream to look for csv files placed into a folder (the five `bikeDetails_add*.csv`
files). When a csv comes in, we want to transform it using the fitted pipeline’s `.transform()` method! A
few notes:
- You’ll need a schema to set up the `readStream`. You can use the SQL data frame’s schema from above!
(`.schema` attribute)
- Each file you’ll be adding to the folder has a header!

We’ll just write the output to the ‘console’ using the ‘append’ mode.\
Once that is all set up, make sure your folder you are looking for the .csv files is empty. Then start the
query. Place the other files into the folder one at a time. You should see the output appear below your
query. Once you’ve done all five files, stop the query!\
This process will set us up well for using a fitted model to obtain predictions on streaming data!

## 1. Read the training CSV
I first read in the `bikeDetails_for_fit.csv` as a spark (SQL) data frame.

In [6]:
bike_df = spark.read.csv("bikeDetails_for_fit.csv", header = True, inferSchema = True)

## 2. Create the SQLTransformer
Then I used an `SQLTransformer` to do log transformation for `km_driven` variable, rename it as `log_km_driven`, and create a dummy variable from `owner` variable coding as 1 = one_owner, and 0 = others.

In [7]:
# import module
from pyspark.ml.feature import SQLTransformer

# conduct transformations
sqlTrans = SQLTransformer(
    statement = """
        SELECT 
            log(selling_price) AS label,
            year, log(km_driven) AS log_km_driven,
            CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
        FROM __THIS__
    """
)

## 3. Create the VectorAssembler
Next, I used the `VectorAssembler` to combine `year`, `log_km_driven`, and `one_owner` into a single feature vecctor stored in the `features` column required by Spark ML.

In [8]:
# import module
from pyspark.ml.feature import VectorAssembler

# Create the vectorAssembler
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

## 4. Create and fit the Pipeline
I then created a Pipeline consisting of the SQLTransformer followed by the VectorAssembler. After defining the two stages, I fit the pipeline to the bikeDetails_for_fit.csv DataFrame to produce a fitted pipeline object that can be applied to new streaming data.

In [9]:
# import module
from pyspark.ml import Pipeline
# built the pipeline
pipeline = Pipeline(stages=[sqlTrans, assembler])

# fit the pipeline on the batch DataFrame
fitted_pipeline = pipeline.fit(bike_df)

## 5. Set up a streaming DataFrame
To prepare the streaming workflow, I used the schema from the fitted DataFrame to define the structure of the incoming CSV files. I then created a `readStream` that monitors a folder for new CSV files, specifying both the schema and `header=True` since each file includes a header row. After setting up the stream, I applied the fitted pipeline’s .`transform()` method to the streaming DataFrame so that each incoming file is automatically processed using the same transformations as the training data.

In [10]:
# set up a steam 
stream_schema = bike_df.schema

# creat the streaming DataFrame
stream_df = (
    spark.readStream
        .schema(stream_schema)
        .option("header", True)
        .csv("stream_input_folder")
)

## 6. Apply the fitted pipeline to the streaming data
Next, I applied the fitted pipeline to the streaming DataFrame, wrote the transformed output to the console (displayed directly in the notebook), and started the streaming query. After the stream was running, I added the five `bikeDetails_add*.csv` files to the `stream_input_folder` one at a time so that each file was automatically processed.

In [11]:
# apply the fitted pipeline to the streaming DataFrame
stream_transformed = fitted_pipeline.transform(stream_df)

# write the streming output to the console and start the stream
query = (
    stream_transformed.writeStream
                      .format("console")
                      .outputMode("append")
                      .start()
)

26/04/19 16:24:34 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3254da62-66b3-4082-bbfc-2dc8cfe599bb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 16:24:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

## 7. Stop the stream
Once all five files had been added to the stream_input_folder, the streaming query processed each file and displayed the transformed output in the notebook. After all batches were processed, I stopped the stream.

In [12]:
query.stop()

26/04/19 16:27:43 WARN DAGScheduler: Failed to cancel job group 03606824-5fee-43fa-80b0-462881bd3f87. Cannot find active jobs for it.
26/04/19 16:27:43 WARN DAGScheduler: Failed to cancel job group 03606824-5fee-43fa-80b0-462881bd3f87. Cannot find active jobs for it.


# Summary
In this assignment, I implemented two complete streaming workflows in Spark. In Part 1, I created a simple rate‑based stream, applied transformations, and wrote the results to an in‑memory table to observe how Spark processes data continuously. In Part 2, I built a batch‑trained ML pipeline using SQL‑based feature engineering and a VectorAssembler, then applied that fitted pipeline to streaming CSV input. By setting up a streaming DataFrame with the correct schema and writing the transformed output to the console, I demonstrated how Spark can apply consistent preprocessing steps to both static and streaming data. 